## 1. Imports

In [1]:
# Standard libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
import os
from pathlib import Path
from sklearn.preprocessing import StandardScaler

sys.path.append(os.path.abspath(".."))

# EarlyFusion models
from EarlyFusion.EarlyFusionMLP import *
from EarlyFusion.EarlyFusionCNN import *
from EarlyFusion.utils import cross_validation_5fold_early_fusion

## 2. Data Loading

In [2]:
AUDIO_FEATURES = "/mloscratch/users/gnahas/data/features/acoustic_features_vf.csv"
LABELS = "/mloscratch/users/mihaylov/NeuroMeditron/src_GAMMA/paired_healthcode.csv"
TRAIN_FOLDS = "/mloscratch/users/gnahas/data/data_paired/5_fold_CV/processed_paired/paired_splits/balanced_train/healthcode_5fold_train.csv"
VAL_TEST_FOLDS = "/mloscratch/users/gnahas/data/data_paired/5_fold_CV/processed_paired/paired_splits/balanced_train/healthcode_5fold_val_test.csv"

## 3. Concatenation

In [3]:
# # Save extracted features as CSV for CNN evaluation
# extracted_features_csv_path = "path/to/concatenated_extracted_features.csv"
# extracted_features_df = pd.DataFrame(extracted_features)
# # extracted_features_df.to_csv(extracted_features_csv_path, index=False)

# print(f"Extracted features saved to: {extracted_features_csv_path}")
# print(f"Extracted features shape: {extracted_features_df.shape}")

## 4. Cross-Validation

In [3]:
# Load the data files
features_df = pd.read_csv(AUDIO_FEATURES)
labels_df = pd.read_csv(LABELS)
train_folds_df = pd.read_csv(TRAIN_FOLDS)
val_test_folds_df = pd.read_csv(VAL_TEST_FOLDS)

print(f"Loaded features: {features_df.shape}")
print(f"  Columns: {list(features_df.columns[:5])}...")

print(f"Loaded labels: {labels_df.shape}")
print(f"  Columns: {list(labels_df.columns)}")

print(f"Train folds: {train_folds_df.shape}")
print(f"  Columns: {list(train_folds_df.columns)}")
print(f"Val+Test folds: {val_test_folds_df.shape}")
print(f"  Columns: {list(val_test_folds_df.columns)}")

# Separate val and test based on 'subset' column
val_folds_df = val_test_folds_df[val_test_folds_df['subset'] == 'val']
test_folds_df = val_test_folds_df[val_test_folds_df['subset'] == 'test']
print(f"  - Validation folds: {val_folds_df.shape}")
print(f"  - Test folds: {test_folds_df.shape}")

if 'healthcode' in features_df.columns and 'healthCode' not in features_df.columns:
        features_df = features_df.rename(columns={'healthcode': 'healthCode'})
        print(f"Renamed 'healthcode' → 'healthCode' for consistency")

# Merge features with labels on healthCode
features_df = features_df.merge(labels_df[['healthCode', 'label_PD']], on='healthCode', how='inner')
print(f"Features after merging with labels: {features_df.shape}")

# Filter to only include healthcodes in 5-fold splits
all_fold_healthcodes = set(train_folds_df['healthCode'].unique()) | set(val_test_folds_df['healthCode'].unique())
features_df = features_df[features_df['healthCode'].isin(all_fold_healthcodes)]
print(f"Features after filtering to 5-fold healthcodes: {features_df.shape}")

# Prepare feature columns (exclude metadata)
# After renaming: 'healthcode' → 'healthCode', so only 'healthCode' exists
metadata_cols = ['filename', 'healthCode', 'record_id', 'label_PD']
feature_cols = [col for col in features_df.columns if col not in metadata_cols]
print(f"Number of features: {len(feature_cols)}")
print(f"Feature columns: {feature_cols[:5]}... (showing first 5)")

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}\n")

# Get unique patient healthCodes for fold 0 (for filtering)
train_patient_ids = train_folds_df[train_folds_df['fold_iteration'] == 0]['healthCode'].values
val_patient_ids = val_folds_df[val_folds_df['fold_iteration'] == 0]['healthCode'].values
test_patient_ids = test_folds_df[test_folds_df['fold_iteration'] == 0]['healthCode'].values

print(f"Train patients: {len(train_patient_ids)}")
print(f"Val patients: {len(val_patient_ids)}")
print(f"Test patients: {len(test_patient_ids)}")

# Split data by patient healthCode
train_data = features_df[features_df['healthCode'].isin(train_patient_ids)]
val_data = features_df[features_df['healthCode'].isin(val_patient_ids)]
test_data = features_df[features_df['healthCode'].isin(test_patient_ids)]

print(f"Train samples (before undersampling): {len(train_data)}")
print(f"Val samples: {len(val_data)}")
print(f"Test samples: {len(test_data)}")

# Extract features and labels
X_train = train_data[feature_cols].values.astype(np.float32)
y_train = train_data['label_PD'].values
X_val = val_data[feature_cols].values.astype(np.float32)
y_val = val_data['label_PD'].values
X_test = test_data[feature_cols].values.astype(np.float32)
y_test = test_data['label_PD'].values

# Handle NaN values
X_train = np.nan_to_num(X_train, nan=0.0, posinf=0.0, neginf=0.0)
X_val = np.nan_to_num(X_val, nan=0.0, posinf=0.0, neginf=0.0)
X_test = np.nan_to_num(X_test, nan=0.0, posinf=0.0, neginf=0.0)

# Normalize features using StandardScaler (fit on train, transform on val/test)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

# Convert to tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.long)

# Initialize model
model = EarlyFusionMLP(
    input_dim=X_train.shape[1],
    hidden_dims=[256, 128, 64],
    dropout=0.3,
    verbose=True
)

# Train model
print("\n" + "="*50)
print("Training EarlyFusionMLP...")
print("="*50)
model.fit(
    X_train_tensor, 
    y_train_tensor,
    val_features=X_val_tensor,
    val_labels=y_val_tensor,
    epochs=100,
    class_weight = 3,
    batch_size=64,
    lr=0.001,
    weight_decay=0.01,
    verbose=True
)

# Test model with majority voting per patient
print("\n" + "="*50)
print("Testing EarlyFusionMLP with Majority Voting...")
print("="*50)

# Prepare test dataframe with standardized features
test_data_standardized = test_data.copy()
test_data_standardized[feature_cols] = X_test

# Use the test_with_majority_counting method
test_results = model.test_with_majority_counting(test_data_standardized)

# Display results
print(f"\nPatient-level Results (Majority Voting):")
print(f"  Number of patients: {test_results['num_patients']}")
print(f"  Number of trials: {test_results['num_trials']}")
print(f"  Test Accuracy: {test_results['test_acc']:.4f}")
print(f"  Test F1 Score: {test_results['test_f1']:.4f}")
print(f"  Test ROC AUC: {test_results['test_roc_auc']:.4f}")
print(f"\nConfusion Matrix:\n{test_results['test_conf_mat']}")

Loaded features: (14948, 157)
  Columns: ['filename', 'healthcode', 'record_id', 'mfcc_1_mean', 'mfcc_1_std']...
Loaded labels: (532, 3)
  Columns: ['healthCode', 'label_PD', 'modality']
Train folds: (1380, 3)
  Columns: ['fold_iteration', 'healthCode', 'subset']
Val+Test folds: (1064, 3)
  Columns: ['fold_iteration', 'healthCode', 'subset']
  - Validation folds: (532, 3)
  - Test folds: (532, 3)
Renamed 'healthcode' → 'healthCode' for consistency
Features after merging with labels: (14944, 158)
Features after filtering to 5-fold healthcodes: (14944, 158)
Number of features: 154
Feature columns: ['mfcc_1_mean', 'mfcc_1_std', 'mfcc_2_mean', 'mfcc_2_std', 'mfcc_3_mean']... (showing first 5)
Using device: cuda

Train patients: 276
Val patients: 107
Test patients: 107
Train samples (before undersampling): 7198
Val samples: 3448
Test samples: 2284
Input dimension: 154
MLP architecture: 154 -> 256 -> 128 -> 64 -> 2 (binary classification)
Device: cuda

Training EarlyFusionMLP...


/mloscratch/users/mihaylov/NeuroMeditron/src_GAMMA/EarlyFusion/EarlyFusionMLP.py:158: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /opt/pytorch/pytorch/aten/src/ATen/native/Scalar.cpp:22.)
  epoch_loss += loss.item() * len(lbl)


Epoch 10/100 | Train Loss: 0.2459 | Train Acc: 0.8990 | Val Loss: 0.9390 | Val Acc: 0.6015 | LR: 0.001000
Learning rate reduced to 0.000500
Epoch 20/100 | Train Loss: 0.1592 | Train Acc: 0.9418 | Val Loss: 1.1152 | Val Acc: 0.6325 | LR: 0.000500
Learning rate reduced to 0.000250
Early stopping triggered at epoch 21. No improvement for 20 epochs.
Training complete. Final - Loss: 0.1497, Acc: 0.9453

Testing EarlyFusionMLP with Majority Voting...

Patient-level Results (Majority Voting):
  Number of patients: 107
  Number of trials: 2284
  Test Accuracy: 0.6168
  Test F1 Score: 0.6612
  Test ROC AUC: 0.6126

Confusion Matrix:
[[26 20]
 [21 40]]


In [4]:
output_dir_mlp = "/mloscratch/users/mihaylov/NeuroMeditron/src_GAMMA/models/EarlyFusion"

print("Running 5-fold CV with EarlyFusionMLP on Extracted Features...")
best_model_path_mlp_features, metrics_history_mlp_features = cross_validation_5fold_early_fusion(
    features_csv=AUDIO_FEATURES,
    labels_csv=LABELS,
    train_folds_csv=TRAIN_FOLDS,
    val_test_folds_csv=VAL_TEST_FOLDS,
    output_dir=output_dir_mlp,
    NN_type="MLP",
    hidden_dims=[512, 256, 128],
    batch_size=64,
    num_epochs=100,
    learning_rate=0.001,
    weight_decay=0.0,
    dropout=0.5,
    verbose=True
)
print(f"Best MLP model (features) saved at: {best_model_path_mlp_features}")

Running 5-fold CV with EarlyFusionMLP on Extracted Features...
5-Fold Cross-Validation with MLP
Input dimension: 154
Hidden dimensions: [512, 256, 128]
Number of samples: 14948
Device: cuda

Fold 1/5
Train samples: 276, Test samples: 107
Input dimension: 154
MLP architecture: 154 -> 512 -> 256 -> 128 -> 1 (binary classification)
Device: cuda
Training MLP model...


TypeError: 'int' object is not callable

## 5. Results Visualization

In [ ]:
# Prepare data for MLP (features) visualization
mlp_folds = metrics_history_mlp_features['folds']
mlp_acc = metrics_history_mlp_features['test_acc']
mlp_f1 = metrics_history_mlp_features['test_f1']
mlp_auc = metrics_history_mlp_features['test_roc_auc']
mlp_loss = metrics_history_mlp_features['test_loss']

print("MLP Results (Extracted Features):")
print(f"  Accuracy: {np.mean(mlp_acc):.4f} ± {np.std(mlp_acc):.4f}")
print(f"  F1 Score: {np.mean(mlp_f1):.4f} ± {np.std(mlp_f1):.4f}")
print(f"  ROC AUC:  {np.mean(mlp_auc):.4f} ± {np.std(mlp_auc):.4f}")
print(f"  Loss:     {np.mean(mlp_loss):.4f} ± {np.std(mlp_loss):.4f}")

In [ ]:
# Create MLP (features) comparison plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Cross-Validation Results: MLP on Extracted Features', fontsize=16, fontweight='bold')

# Plot 1: Accuracy
axes[0, 0].plot(mlp_folds, mlp_acc, marker='s', linewidth=2, markersize=8, color='steelblue')
axes[0, 0].axhline(np.mean(mlp_acc), color='steelblue', linestyle='--', alpha=0.5)
axes[0, 0].fill_between(mlp_folds, np.mean(mlp_acc) - np.std(mlp_acc), 
                         np.mean(mlp_acc) + np.std(mlp_acc), alpha=0.2, color='steelblue')
axes[0, 0].set_xlabel('Fold')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].set_title('Test Accuracy per Fold')
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: F1 Score
axes[0, 1].plot(mlp_folds, mlp_f1, marker='s', linewidth=2, markersize=8, color='darkgreen')
axes[0, 1].axhline(np.mean(mlp_f1), color='darkgreen', linestyle='--', alpha=0.5)
axes[0, 1].fill_between(mlp_folds, np.mean(mlp_f1) - np.std(mlp_f1), 
                         np.mean(mlp_f1) + np.std(mlp_f1), alpha=0.2, color='darkgreen')
axes[0, 1].set_xlabel('Fold')
axes[0, 1].set_ylabel('F1 Score')
axes[0, 1].set_title('Test F1 Score per Fold')
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: ROC AUC
axes[1, 0].plot(mlp_folds, mlp_auc, marker='s', linewidth=2, markersize=8, color='darkorange')
axes[1, 0].axhline(np.mean(mlp_auc), color='darkorange', linestyle='--', alpha=0.5)
axes[1, 0].fill_between(mlp_folds, np.mean(mlp_auc) - np.std(mlp_auc), 
                         np.mean(mlp_auc) + np.std(mlp_auc), alpha=0.2, color='darkorange')
axes[1, 0].set_xlabel('Fold')
axes[1, 0].set_ylabel('ROC AUC')
axes[1, 0].set_title('Test ROC AUC per Fold')
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Loss
axes[1, 1].plot(mlp_folds, mlp_loss, marker='s', linewidth=2, markersize=8, color='crimson')
axes[1, 1].axhline(np.mean(mlp_loss), color='crimson', linestyle='--', alpha=0.5)
axes[1, 1].fill_between(mlp_folds, np.mean(mlp_loss) - np.std(mlp_loss), 
                         np.mean(mlp_loss) + np.std(mlp_loss), alpha=0.2, color='crimson')
axes[1, 1].set_xlabel('Fold')
axes[1, 1].set_ylabel('Loss')
axes[1, 1].set_title('Test Loss per Fold')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()